# pySCENIC

### Нужно 4 Gb RAM на 1 worker минимум! Лучше 8 для этапа GRN! 1 worker =  1 поток CPU

### Оптимум Gb RAM = n CPU Cores * 2 * 8

In [1]:
import pyscenic
import json
import zlib
import base64
import loompy as lp
import pandas as pd

from dask.distributed import Client, LocalCluster
from arboreto.algo import grnboost2
from arboreto.utils import load_tf_names

from pyscenic.binarization import binarize

/Users/CatPro/miniconda3/envs/3.10/lib/python3.10/site-packages/loompy/bus_file.py:68: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  def twobit_to_dna(twobit: int, size: int) -> str:
/Users/CatPro/miniconda3/envs/3.10/lib/python3.10/site-packages/loompy/bus_file.py:85: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  def dna_to_twobit(dna: str) -> int:
/Users/CatPr

In [2]:
print("Extracting the expression matrix from the Loom file...")
with lp.connect("Seurat.loom", mode="r", validate=False) as ds:

    pdf = pd.DataFrame(ds[:, :].T, index=ds.ca.CellID, columns=ds.ra.Gene)

pdf.columns = pdf.columns.astype(str)

print("Loading Transcription Factor (TF) names...")
tfs = load_tf_names("mm_mgi_tfs.txt")

print("Initializing Dask LocalCluster (Shared Memory Mode)...")
cluster = LocalCluster(
    n_workers=1, 
    threads_per_worker=22,
    processes=False 
)
client = Client(cluster)

try:
    print("Launching GRNBoost2 (Inferring regulatory networks)...")
    network = grnboost2(
        expression_data=pdf, 
        tf_names=tfs, 
        client_or_address=client
    )
    
    print("Saving the inferred network to CSV...")
    network.to_csv("adj.csv", index=False)
    print("Success! The results have been saved to 'adj.csv'")
    
finally:
    # Clean up and release system resources gracefully
    print("Shutting down the Dask cluster...")
    client.close()
    cluster.close()

Extracting the expression matrix from the Loom file...
Loading Transcription Factor (TF) names...
Initializing Dask LocalCluster (Shared Memory Mode)...
Launching GRNBoost2 (Inferring regulatory networks)...


/Users/CatPro/miniconda3/envs/3.10/lib/python3.10/site-packages/distributed/client.py:3108: UserWarning: Sending large graph of size 433.06 MiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(


Saving the inferred network to CSV...
Success! The results have been saved to 'adj.csv'
Shutting down the Dask cluster...


In [3]:
# CLI 

!pyscenic ctx adj.csv \
    *feather \
    --annotations_fname motifs-v10nr_clust-nr.mgi-m0.001-o0.0.tbl \
    --expression_mtx_fname Seurat.loom \
    --output reg.csv \
    --mask_dropouts \
    --num_workers 16

/Users/CatPro/miniconda3/envs/3.10/lib/python3.10/site-packages/loompy/bus_file.py:68: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  def twobit_to_dna(twobit: int, size: int) -> str:
/Users/CatPro/miniconda3/envs/3.10/lib/python3.10/site-packages/loompy/bus_file.py:85: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  def dna_to_twobit(dna: str) -> int:
/Users/CatPr

In [4]:
# AUCell 

!pyscenic aucell Seurat.loom reg.csv --output Seurat_out.loom --num_workers 16

/Users/CatPro/miniconda3/envs/3.10/lib/python3.10/site-packages/loompy/bus_file.py:68: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  def twobit_to_dna(twobit: int, size: int) -> str:
/Users/CatPro/miniconda3/envs/3.10/lib/python3.10/site-packages/loompy/bus_file.py:85: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  def dna_to_twobit(dna: str) -> int:
/Users/CatPr

In [5]:
# Binarize the AUCell output

print("Extracting AUC matrix from Loom...")

with lp.connect("Seurat_out.loom", mode='r', validate=False) as lf:
    auc_mtx = pd.DataFrame(lf.ca.RegulonsAUC, index=lf.ca.CellID)

auc_mtx.index.names = ['Cells']

print("Saving raw AUC matrix...")
auc_mtx.to_csv('pySCENIC-AUC-Raw.csv')

print("Binarizing the AUC matrix...")
auc_mtx_binary, thresholds = binarize(auc_mtx, num_workers=22)

print("Saving binary AUC matrix...")
df_auc_mtx_binary = pd.DataFrame(auc_mtx_binary)
df_auc_mtx_binary.index.names = ['Cells']

# Export the binary matrix
df_auc_mtx_binary.to_csv('pySCENIC-AUC-Binary.csv')

print("Success! Both 'pySCENIC-AUC-Raw.csv' and 'pySCENIC-AUC-Binary.csv' are ready.")

Extracting AUC matrix from Loom...
Saving raw AUC matrix...
Binarizing the AUC matrix...
Saving binary AUC matrix...
Success! Both 'pySCENIC-AUC-Raw.csv' and 'pySCENIC-AUC-Binary.csv' are ready.
